# NSE 500% Forward-Return Research — DuckDB Optimized

This rewrite keeps the original research objective but moves the heavy data work from pandas into DuckDB.

**Optimization strategy:** scan the Parquet universe in DuckDB only to discover +500% forward-return events. Once qualifying winner symbols are known, calculate features and trajectories only for those symbols.

Source: fileciteturn0file0


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, time, warnings
warnings.filterwarnings("ignore")

DATA_DIR = Path("/content/drive/MyDrive/quant/data/parquet/daily")
OUTPUT_DIR = Path("/content/drive/MyDrive/quant/results/500pct_price_structure_duckdb")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FORWARD_DAYS = 252
WINNER_RETURN = 5.0
MIN_HISTORY = 252

APPLY_LIQUIDITY_FILTER = False
MIN_MEDIAN_DAILY_TURNOVER_60D = 1_000_000

TRAJ_PRE_DAYS = 120
TRAJ_POST_DAYS = 252

print("Parquet files:", len(list(DATA_DIR.rglob("*.parquet"))))


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb", "pyarrow"])

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

con = duckdb.connect(":memory:")
con.execute("PRAGMA threads=8")
con.execute("PRAGMA enable_progress_bar=true")

files = sorted(DATA_DIR.rglob("*.parquet"))
if not files:
    raise FileNotFoundError(DATA_DIR)

print("DuckDB:", duckdb.__version__)
print("Files:", len(files))


In [ ]:
# Infer the input columns from one Parquet file.
sample = str(files[0]).replace("'", "''")
schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{sample}')"
).df()

cols = schema["column_name"].tolist()
norm = {str(c).strip().lower(): c for c in cols}

ALIASES = {
    "date": ["date","datetime","timestamp","trade_date","dt"],
    "symbol": ["symbol","ticker","tradingsymbol","security","stock","name"],
    "open": ["open","open_price"],
    "high": ["high","high_price"],
    "low": ["low","low_price"],
    "close": ["close","close_price","last","ltp"],
    "volume": ["volume","vol","total_traded_qty","quantity"],
}

def resolve(aliases):
    for x in aliases:
        if x in norm:
            return norm[x]
    return None

resolved = {k: resolve(v) for k,v in ALIASES.items()}
missing = [x for x in ["date","symbol","open","high","low","close"] if resolved[x] is None]
if missing:
    raise ValueError(f"Missing columns: {missing}; available={cols}")

print(resolved)


In [ ]:
# Lazy normalized Parquet view.
def qi(x):
    return '"' + str(x).replace('"','""') + '"'

def cast_expr(target, source):
    if source is None:
        return "CAST(NULL AS DOUBLE)" if target == "volume" else "CAST(NULL AS VARCHAR)"
    if target == "date":
        return f"try_cast({qi(source)} AS DATE)"
    if target == "symbol":
        return f"upper(trim(cast({qi(source)} AS VARCHAR)))"
    return f"try_cast({qi(source)} AS DOUBLE)"

glob_path = str(DATA_DIR).replace("'","''") + "/**/*.parquet"

select_sql = ",\n".join(
    f"    {cast_expr(k, resolved[k])} AS {k}"
    for k in ["date","symbol","open","high","low","close","volume"]
)

con.execute(f"""
CREATE OR REPLACE VIEW daily_raw AS
SELECT
{select_sql}
FROM read_parquet(
    '{glob_path}',
    union_by_name=true,
    hive_partitioning=true
)
""")

# Deduplication is also performed in DuckDB.
con.execute("""
CREATE OR REPLACE VIEW daily AS
SELECT date, symbol, open, high, low, close, volume
FROM (
    SELECT *,
           row_number() OVER (
               PARTITION BY symbol,date
               ORDER BY close DESC NULLS LAST
           ) AS rn
    FROM daily_raw
    WHERE date IS NOT NULL
      AND symbol IS NOT NULL
      AND close IS NOT NULL
      AND close > 0
)
WHERE rn=1
""")

display(con.execute("""
SELECT COUNT(*) rows,
       COUNT(DISTINCT symbol) symbols,
       MIN(date) first_date,
       MAX(date) last_date
FROM daily
""").df())


## Find the 500% candidates

This broad scan is unavoidable because the program must inspect the universe to know which stocks eventually reach 6× within 252 trading sessions.

The important optimization is that **the complete dataset never becomes a pandas DataFrame**.


In [ ]:
t0 = time.time()

con.execute(f"""
CREATE OR REPLACE TEMP VIEW labeled AS
WITH x AS (
    SELECT *,
           row_number() OVER (
               PARTITION BY symbol ORDER BY date
           ) AS obs_num,
           count(*) OVER (
               PARTITION BY symbol ORDER BY date
               ROWS BETWEEN 1 FOLLOWING AND {FORWARD_DAYS} FOLLOWING
           ) AS future_rows,
           max(close) OVER (
               PARTITION BY symbol ORDER BY date
               ROWS BETWEEN 1 FOLLOWING AND {FORWARD_DAYS} FOLLOWING
           ) AS future_max_close
    FROM daily
),
e AS (
    SELECT *,
           future_max_close/close-1 AS future_return_max,
           future_max_close/close-1 >= {WINNER_RETURN} AS winner
    FROM x
    WHERE obs_num > {MIN_HISTORY}
      AND future_rows = {FORWARD_DAYS}
),
l AS (
    SELECT *,
           lag(winner) OVER (
               PARTITION BY symbol ORDER BY date
           ) AS previous_winner
    FROM e
)
SELECT *,
       winner AND NOT coalesce(previous_winner,false) AS winner_run_start
FROM l
""")

con.execute("""
CREATE OR REPLACE TEMP VIEW first_events AS
SELECT *
FROM labeled
WHERE winner_run_start
""")

first_events = con.execute("""
SELECT symbol,date,close,future_max_close,future_return_max
FROM first_events
ORDER BY future_return_max DESC
""").df()

print(f"Winner discovery time: {time.time()-t0:.1f}s")
print("Independent winner events:", len(first_events))
print("Winner symbols:", first_events.symbol.nunique())
display(first_events.head(30))


In [ ]:
# From here onward, the expensive feature pipeline is restricted to winner symbols.
con.execute("""
CREATE OR REPLACE TEMP VIEW candidate_symbols AS
SELECT DISTINCT symbol FROM first_events
""")

print(
    "Candidate symbols:",
    con.execute("SELECT COUNT(*) FROM candidate_symbols").fetchone()[0]
)


## Candidate-only feature engine

The old notebook calculated rolling features for every stock. This version calculates them only for symbols that produced a qualifying 500% event.

DuckDB handles the rolling windows, joins and derived columns.


In [ ]:
con.execute("""
CREATE OR REPLACE TEMP VIEW candidate_history AS
SELECT
    d.*,
    row_number() OVER (PARTITION BY d.symbol ORDER BY d.date) AS obs_num,
    lag(d.close) OVER (PARTITION BY d.symbol ORDER BY d.date) AS prev_close,
    lag(d.close,5) OVER (PARTITION BY d.symbol ORDER BY d.date) AS close_5,
    lag(d.close,20) OVER (PARTITION BY d.symbol ORDER BY d.date) AS close_20,
    lag(d.close,60) OVER (PARTITION BY d.symbol ORDER BY d.date) AS close_60,
    lag(d.close,120) OVER (PARTITION BY d.symbol ORDER BY d.date) AS close_120,
    lag(d.close,252) OVER (PARTITION BY d.symbol ORDER BY d.date) AS close_252,
    greatest(
        d.high-d.low,
        abs(d.high-lag(d.close) OVER (PARTITION BY d.symbol ORDER BY d.date)),
        abs(d.low-lag(d.close) OVER (PARTITION BY d.symbol ORDER BY d.date))
    ) AS tr,
    ln(
        d.close/nullif(lag(d.close) OVER (PARTITION BY d.symbol ORDER BY d.date),0)
    ) AS logret
FROM daily d
JOIN candidate_symbols s USING(symbol)
""")

# Rolling primitives.
con.execute("""
CREATE OR REPLACE TEMP VIEW f1 AS
SELECT *,
    avg(close) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 49 PRECEDING AND CURRENT ROW) AS sma_50,
    avg(close) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 99 PRECEDING AND CURRENT ROW) AS sma_100,
    avg(close) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 199 PRECEDING AND CURRENT ROW) AS sma_200,

    avg(tr) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 13 PRECEDING AND CURRENT ROW) AS atr_14,

    stddev_samp(logret) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)*sqrt(252) AS volatility_20d,
    stddev_samp(logret) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)*sqrt(252) AS volatility_60d,
    stddev_samp(logret) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW)*sqrt(252) AS volatility_120d,

    max(high) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING) AS prior_high_20d,
    max(high) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 60 PRECEDING AND 1 PRECEDING) AS prior_high_60d,
    max(high) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 120 PRECEDING AND 1 PRECEDING) AS prior_high_120d,
    max(high) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 252 PRECEDING AND 1 PRECEDING) AS prior_high_252d,

    min(low) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING) AS prior_low_20d,
    min(low) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 60 PRECEDING AND 1 PRECEDING) AS prior_low_60d,
    min(low) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 120 PRECEDING AND 1 PRECEDING) AS prior_low_120d,
    min(low) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 252 PRECEDING AND 1 PRECEDING) AS prior_low_252d,

    avg((high-low)/nullif(close,0)) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS avg_range_20d,
    avg((high-low)/nullif(close,0)) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW) AS avg_range_60d,

    avg(volume) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS volume_20d,
    avg(volume) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW) AS volume_60d,
    avg(volume) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS volume_5d
FROM candidate_history
""")

# Momentum and moving-average slopes.
con.execute("""
CREATE OR REPLACE TEMP VIEW f2 AS
SELECT *,
    close/NULLIF(close_5,0)-1 AS ret_5d,
    close/NULLIF(close_20,0)-1 AS ret_20d,
    close/NULLIF(close_60,0)-1 AS ret_60d,
    close/NULLIF(close_120,0)-1 AS ret_120d,
    close/NULLIF(close_252,0)-1 AS ret_252d,

    sma_50/NULLIF(lag(sma_50,20) OVER (PARTITION BY symbol ORDER BY date),0)-1 AS sma_50_slope_20d,
    sma_100/NULLIF(lag(sma_100,20) OVER (PARTITION BY symbol ORDER BY date),0)-1 AS sma_100_slope_20d,
    sma_200/NULLIF(lag(sma_200,20) OVER (PARTITION BY symbol ORDER BY date),0)-1 AS sma_200_slope_20d,

    volume/NULLIF(volume_20d,0) AS volume_vs_20d,
    volume/NULLIF(volume_60d,0) AS volume_vs_60d,
    volume_20d/NULLIF(volume_60d,0)-1 AS volume_trend_20_vs_60
FROM f1
""")

# Derived features.
con.execute("""
CREATE OR REPLACE TEMP VIEW candidate_features AS
SELECT *,
    close/NULLIF(sma_50,0)-1 AS close_sma_50_ratio,
    close/NULLIF(sma_100,0)-1 AS close_sma_100_ratio,
    close/NULLIF(sma_200,0)-1 AS close_sma_200_ratio,

    (prior_high_20d-prior_low_20d)/NULLIF(close,0) AS range_20d_pct,
    (prior_high_60d-prior_low_60d)/NULLIF(close,0) AS range_60d_pct,
    (prior_high_120d-prior_low_120d)/NULLIF(close,0) AS range_120d_pct,
    (prior_high_252d-prior_low_252d)/NULLIF(close,0) AS range_252d_pct,

    close/NULLIF(prior_high_60d,0)-1 AS distance_prior_high_60d,
    close/NULLIF(prior_high_120d,0)-1 AS distance_prior_high_120d,
    close/NULLIF(prior_high_252d,0)-1 AS distance_prior_high_252d,

    (close-prior_low_60d)/NULLIF(prior_high_60d-prior_low_60d,0) AS range_position_60d,
    (close-prior_low_120d)/NULLIF(prior_high_120d-prior_low_120d,0) AS range_position_120d,
    (close-prior_low_252d)/NULLIF(prior_high_252d-prior_low_252d,0) AS range_position_252d,

    close/NULLIF(prior_high_252d,0)-1 AS drawdown_from_high_252d,

    (prior_high_60d-prior_low_60d)/NULLIF(close,0) /
      NULLIF((prior_high_120d-prior_low_120d)/NULLIF(close,0),0) AS range60_to_range120,

    avg_range_20d/NULLIF(avg_range_60d,0) AS range_tightening_20_vs_60,

    atr_14/NULLIF(close,0) AS atr_14_pct,

    (prior_high_252d-close)/NULLIF(atr_14,0) AS distance_252d_high_atr,

    CASE WHEN sma_50>sma_100 THEN 1 ELSE 0 END AS sma_50_gt_100,
    CASE WHEN sma_100>sma_200 THEN 1 ELSE 0 END AS sma_100_gt_200,
    CASE WHEN prior_low_20d>prior_low_60d THEN 1 ELSE 0 END AS higher_low_20_vs_60,
    CASE WHEN prior_high_20d>prior_high_60d THEN 1 ELSE 0 END AS higher_high_20_vs_60,

    sum(CASE WHEN close>=prior_high_60d*0.95 THEN 1 ELSE 0 END)
      OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)
      AS days_near_60d_high_20d,

    sum(CASE WHEN close>=prior_high_252d*0.90 THEN 1 ELSE 0 END)
      OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
      AS days_near_252d_high_60d,

    median(close*volume) OVER (
      PARTITION BY symbol ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW
    ) AS median_turnover_60d
FROM f2
""")

# Final derived values.
con.execute("""
CREATE OR REPLACE TEMP VIEW candidate_features_final AS
SELECT *,
    ret_20d-lag(ret_20d,20) OVER (
        PARTITION BY symbol ORDER BY date
    ) AS momentum_acceleration
FROM candidate_features
""")

print("Candidate-only features ready.")


In [ ]:
# Join features only to the independent winner events.
liquidity = (
    f"AND f.median_turnover_60d >= {MIN_MEDIAN_DAILY_TURNOVER_60D}"
    if APPLY_LIQUIDITY_FILTER else ""
)

con.execute(f"""
CREATE OR REPLACE TEMP VIEW winner_dataset AS
SELECT
    e.symbol,
    e.date,
    e.close,
    e.future_max_close,
    e.future_return_max,
    e.winner,
    f.* EXCLUDE(symbol,date,close)
FROM first_events e
JOIN candidate_features_final f USING(symbol,date)
WHERE f.obs_num > {MIN_HISTORY}
{liquidity}
""")

winner_dataset = con.execute("""
SELECT *
FROM winner_dataset
ORDER BY future_return_max DESC
""").df()

print("Winner feature rows:", len(winner_dataset))
display(winner_dataset.head())


## Optional comparison

To preserve the original idea without rebuilding the entire market feature matrix, the comparison uses non-winner observations from the **same symbols that produced a winner**. This is much cheaper but is not a market-wide control sample.


In [ ]:
comparison_features = [
    "ret_20d","ret_60d","ret_120d","ret_252d",
    "close_sma_50_ratio","close_sma_100_ratio","close_sma_200_ratio",
    "sma_50_slope_20d","sma_100_slope_20d","sma_200_slope_20d",
    "range_60d_pct","range_120d_pct","range_252d_pct",
    "range_position_60d","range_position_120d","range_position_252d",
    "distance_prior_high_60d","distance_prior_high_120d","distance_prior_high_252d",
    "atr_14_pct","volatility_20d","volatility_60d",
    "volume_vs_20d","volume_vs_60d","volume_trend_20_vs_60",
    "days_near_252d_high_60d","distance_252d_high_atr"
]

con.execute("""
CREATE OR REPLACE TEMP VIEW candidate_labeled_features AS
SELECT
    l.*,
    f.* EXCLUDE(symbol,date,close)
FROM labeled l
JOIN candidate_features_final f USING(symbol,date)
""")

rows = []

for feature in comparison_features:
    if feature not in winner_dataset.columns:
        continue

    w = winner_dataset[feature].dropna()

    n = con.execute(f"""
        SELECT {feature}
        FROM candidate_labeled_features
        WHERE NOT winner
          AND {feature} IS NOT NULL
    """).df()[feature]

    if len(w) >= 5 and len(n) >= 5:
        wm, nm = w.median(), n.median()
        rows.append({
            "feature": feature,
            "winner_n": len(w),
            "nonwinner_n": len(n),
            "winner_median": wm,
            "nonwinner_median": nm,
            "winner_mean": w.mean(),
            "nonwinner_mean": n.mean(),
            "median_difference": wm-nm,
            "median_ratio": wm/nm if nm != 0 else np.nan
        })

comparison = pd.DataFrame(rows)

if not comparison.empty:
    comparison = comparison.sort_values(
        "median_difference",
        key=lambda x: x.abs(),
        ascending=False
    )
    display(comparison.head(30))


## Winner-aligned trajectories

Only winner symbols are queried. DuckDB uses trading-session indexes, so `-120..+252` means trading observations rather than calendar days.


In [ ]:
con.execute("""
CREATE OR REPLACE TEMP VIEW indexed_candidates AS
SELECT *,
       row_number() OVER (
           PARTITION BY symbol ORDER BY date
       ) AS session_idx
FROM candidate_features_final
""")

con.execute("""
CREATE OR REPLACE TEMP VIEW event_rows AS
SELECT
    e.symbol,
    e.date AS breakout_date,
    c.session_idx AS event_idx,
    c.close AS event_close
FROM first_events e
JOIN indexed_candidates c USING(symbol,date)
""")

con.execute(f"""
CREATE OR REPLACE TEMP VIEW trajectory_raw AS
SELECT
    e.symbol AS winner_symbol,
    e.breakout_date,
    c.date,
    c.close,
    c.volume,
    c.session_idx-e.event_idx AS relative_day,
    c.close/NULLIF(e.event_close,0)*100 AS price_index_100,
    c.volatility_20d,
    c.drawdown_from_high_252d
FROM event_rows e
JOIN indexed_candidates c
  ON c.symbol=e.symbol
 AND c.session_idx BETWEEN
     e.event_idx-{TRAJ_PRE_DAYS}
     AND e.event_idx+{TRAJ_POST_DAYS}
""")

con.execute(f"""
CREATE OR REPLACE TEMP VIEW trajectory AS
WITH x AS (
    SELECT *,
           median(volume) FILTER (
               WHERE relative_day BETWEEN -{TRAJ_PRE_DAYS} AND -1
           ) OVER (
               PARTITION BY winner_symbol,breakout_date
           ) AS pre_volume_median
    FROM trajectory_raw
)
SELECT *,
       volume/NULLIF(pre_volume_median,0)
         AS volume_vs_prebreakout_median
FROM x
""")

trajectory = con.execute("""
SELECT *
FROM trajectory
ORDER BY winner_symbol,breakout_date,relative_day
""").df()

print("Trajectory rows:", len(trajectory))
display(trajectory.head())


In [ ]:
median_trajectory = con.execute("""
SELECT
    relative_day,
    COUNT(DISTINCT winner_symbol || '|' || CAST(breakout_date AS VARCHAR)) AS winners,
    median(price_index_100) AS median_price_index,
    median(volume_vs_prebreakout_median) AS median_volume_multiple,
    median(volatility_20d) AS median_volatility,
    median(drawdown_from_high_252d) AS median_drawdown,
    quantile_cont(price_index_100,0.25) AS p25_price_index,
    quantile_cont(price_index_100,0.75) AS p75_price_index
FROM trajectory
GROUP BY relative_day
ORDER BY relative_day
""").df()

display(
    median_trajectory[
        median_trajectory.relative_day.isin(
            [-120,-90,-60,-30,-10,-5,-1,0,1,5,10,20,60,120,180,252]
        )
    ]
)


In [ ]:
def plot_metric(column, title, ylabel, filename, hline=None):
    plt.figure(figsize=(13,6))
    plt.plot(
        median_trajectory.relative_day,
        median_trajectory[column],
        linewidth=2
    )
    plt.axvline(0,linestyle="--",linewidth=1)
    if hline is not None:
        plt.axhline(hline,linestyle=":",linewidth=1)
    plt.title(title)
    plt.xlabel("Trading days relative to winner event")
    plt.ylabel(ylabel)
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/filename,dpi=160,bbox_inches="tight")
    plt.show()

plot_metric("median_price_index",
            "500% Winners — Median Price",
            "Price index (t=0 = 100)",
            "median_price_trajectory.png",100)

plot_metric("median_volume_multiple",
            "500% Winners — Median Volume",
            "Volume / pre-event median",
            "median_volume_trajectory.png",1)

plot_metric("median_volatility",
            "500% Winners — Median Volatility",
            "20-day annualized volatility",
            "median_volatility_trajectory.png")

plot_metric("median_drawdown",
            "500% Winners — Median Drawdown",
            "Drawdown from prior 252-day high",
            "median_drawdown_trajectory.png",0)


In [ ]:
# Export compact results.
winner_dataset.to_csv(
    OUTPUT_DIR/"winner_500pct_events.csv", index=False
)
winner_dataset.to_parquet(
    OUTPUT_DIR/"winner_500pct_events.parquet", index=False
)

comparison.to_csv(
    OUTPUT_DIR/"winner_vs_nonwinner_feature_comparison.csv",
    index=False
)

trajectory.to_parquet(
    OUTPUT_DIR/"breakout_aligned_trajectories.parquet",
    index=False
)
trajectory.to_csv(
    OUTPUT_DIR/"breakout_aligned_trajectories.csv",
    index=False
)

median_trajectory.to_csv(
    OUTPUT_DIR/"median_breakout_trajectories.csv",
    index=False
)

summary = con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT symbol) AS symbols,
    MIN(date) AS start_date,
    MAX(date) AS end_date
FROM daily
""").df().iloc[0].to_dict()

summary.update({
    "parquet_files": len(files),
    "forward_days": FORWARD_DAYS,
    "winner_return": WINNER_RETURN,
    "winner_events": len(first_events),
    "winner_symbols": int(first_events.symbol.nunique()),
    "engine": "DuckDB",
    "full_market_pandas_load": False,
    "output_directory": str(OUTPUT_DIR)
})

with open(OUTPUT_DIR/"research_summary.json","w") as f:
    json.dump(summary,f,indent=2,default=str)

print(json.dumps(summary,indent=2,default=str))


## Why this should be substantially faster

- No `pd.concat()` of all Parquet files.
- No full-market pandas DataFrame.
- No pandas `groupby().apply()` for forward labels.
- Forward 252-session maximum is a DuckDB window query.
- Winner-run detection is DuckDB.
- Rolling features are calculated only for winner symbols.
- Trajectory extraction is restricted to winner events.
- Pandas is used only for the reduced outputs.

### Next step for even more speed

For repeated daily runs, build a normalized incremental Parquet/DuckDB layer. Then the daily job can process only new NSE sessions instead of repeatedly parsing the entire historical collection.
